# Layer Normalization: Building Strong Intuitions

**Learning Objectives:**
1. Understand **why** Layer Normalization normalizes across features (not batches)
2. Build strong intuition for **batch independence** - the key differentiator
3. Learn **when to use LayerNorm vs BatchNorm** with a clear decision framework
4. Understand how LayerNorm enables **Transformers** and modern architectures
5. Gain practical implementation and debugging skills

**What makes this notebook different:**
- Theory before practice for every concept
- Progressive visualizations building intuition
- Constant comparison with Batch Normalization
- Hands-on experiments to cement understanding
- Clear guidance on when to use what

Let's dive in! 🚀

In [ ]:
# Standard imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, TensorDataset

# Shared utilities
from aiml_notebooks import get_device, set_seed

# Set style for better plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Reproducibility
set_seed(42)

# Device
device = get_device()

print("Setup complete!")

---
## Part 1: Motivation - Why Not Just Use Batch Normalization?

Before we dive into Layer Normalization, let's understand **why we need it**.

### The Problem with Batch Normalization

Batch Normalization has been incredibly successful, but it has some limitations:

1. **Batch Dependency**: It normalizes across the batch dimension, so statistics depend on which samples are in the batch
2. **Small Batch Issues**: With small batches, statistics become noisy and unreliable
3. **Online Learning**: Difficult to use when processing one sample at a time
4. **Sequence Models**: Awkward for variable-length sequences (RNNs, Transformers)
5. **Train/Test Discrepancy**: Uses batch statistics during training, running averages during testing

### The Key Question

Batch Normalization asks: **"How does this feature compare across samples in the batch?"**

Layer Normalization asks: **"How does each feature of this sample compare to other features in the same sample?"**

This seemingly simple change has profound implications! Let's explore...

### Visual Setup: Understanding Dimensions

Let's create a simple dataset to visualize the difference:
- Batch size: 4 samples
- Features: 3 features per sample

We'll visualize which dimensions each normalization method operates on.

In [ ]:
# Create a simple example batch
# Shape: (batch_size=4, features=3)
example_batch = torch.tensor([
    [1.0, 2.0, 3.0],   # Sample 1
    [4.0, 5.0, 6.0],   # Sample 2
    [7.0, 8.0, 9.0],   # Sample 3
    [10.0, 11.0, 12.0] # Sample 4
])

print("Example Batch:")
print(example_batch)
print(f"\nShape: {example_batch.shape} (batch_size=4, features=3)")

# Visualize as heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: BatchNorm visualization (normalizes across batch dimension)
axes[0].imshow(example_batch.numpy(), cmap='YlOrRd', aspect='auto')
axes[0].set_xlabel('Features', fontsize=12)
axes[0].set_ylabel('Samples (Batch)', fontsize=12)
axes[0].set_title('Batch Normalization\n(Normalizes ↓ across samples)', fontsize=14, fontweight='bold')
axes[0].set_xticks([0, 1, 2])
axes[0].set_yticks([0, 1, 2, 3])

# Add arrows to show normalization direction
axes[0].annotate('', xy=(0.5, 3.5), xytext=(0.5, -0.5),
                arrowprops=dict(arrowstyle='<->', color='blue', lw=3))

# Right: LayerNorm visualization (normalizes across feature dimension)
axes[1].imshow(example_batch.numpy(), cmap='YlOrRd', aspect='auto')
axes[1].set_xlabel('Features', fontsize=12)
axes[1].set_ylabel('Samples (Batch)', fontsize=12)
axes[1].set_title('Layer Normalization\n(Normalizes → across features)', fontsize=14, fontweight='bold')
axes[1].set_xticks([0, 1, 2])
axes[1].set_yticks([0, 1, 2, 3])

# Add arrows to show normalization direction
axes[1].annotate('', xy=(2.5, 0.5), xytext=(-0.5, 0.5),
                arrowprops=dict(arrowstyle='<->', color='green', lw=3))

# Add values as text
for i in range(4):
    for j in range(3):
        for ax in axes:
            ax.text(j, i, f'{example_batch[i, j]:.0f}', 
                   ha='center', va='center', color='black', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n🔑 Key Insight:")
print("   BatchNorm: Each feature normalized independently across batch (↓)")
print("   LayerNorm: Each sample normalized independently across features (→)")

**🤔 Reflection Question:** 

If LayerNorm normalizes each sample independently, what does this imply about batch size?

<details>
<summary>Click to reveal answer</summary>

LayerNorm should be **completely independent of batch size**! Whether you process 1 sample or 1000, each sample is normalized the same way. This is a huge advantage over BatchNorm.

</details>

---
## Part 2: The Core Idea - Per-Sample Normalization

### Intuition Before Math

Imagine you're analyzing student test scores:

**Batch Normalization thinking:**
- "How did student A's math score compare to all other students' math scores?"
- Compares each subject across students

**Layer Normalization thinking:**
- "How did student A's math score compare to their own English and Science scores?"
- Compares all subjects within one student

### Why This Matters

Layer Normalization:
1. Makes each sample's features have **similar scales**
2. **Independent of other samples** in the batch
3. Works great when features might have **very different ranges**
4. Perfect for **sequence models** where each position is somewhat independent

Let's see this in action!

In [ ]:
# Create a more realistic example: features with different scales
# Sample 1: All features are small
# Sample 2: All features are large
# Sample 3: Mixed

scaled_batch = torch.tensor([
    [0.5, 1.0, 1.5],      # Sample 1: small values
    [10.0, 20.0, 30.0],   # Sample 2: large values  
    [2.0, 100.0, 5.0],    # Sample 3: mixed scales
    [15.0, 18.0, 21.0],   # Sample 4: medium values
])

print("Batch with different scales:")
print(scaled_batch)

# Compute statistics for each method
print("\n" + "="*60)
print("BATCH NORMALIZATION (across samples, per feature):")
print("="*60)
for feat_idx in range(3):
    feature_col = scaled_batch[:, feat_idx]
    mean = feature_col.mean()
    std = feature_col.std(unbiased=False)
    print(f"Feature {feat_idx}: mean={mean:.2f}, std={std:.2f}")
    print(f"  Values: {feature_col.tolist()}")

print("\n" + "="*60)
print("LAYER NORMALIZATION (across features, per sample):")
print("="*60)
for sample_idx in range(4):
    sample_row = scaled_batch[sample_idx, :]
    mean = sample_row.mean()
    std = sample_row.std(unbiased=False)
    print(f"Sample {sample_idx}: mean={mean:.2f}, std={std:.2f}")
    print(f"  Values: {sample_row.tolist()}")

print("\n🔑 Notice: LayerNorm stats are computed PER SAMPLE, independent of batch!")

---
## Part 3: The Mathematics - Step by Step

Now that we have intuition, let's formalize it.

### Layer Normalization Formula

Given an input $\mathbf{x} \in \mathbb{R}^{d}$ (a single sample with $d$ features):

$$
\begin{align}
\mu &= \frac{1}{d} \sum_{i=1}^{d} x_i \quad \text{(mean across features)} \\
\sigma^2 &= \frac{1}{d} \sum_{i=1}^{d} (x_i - \mu)^2 \quad \text{(variance across features)} \\
\hat{x}_i &= \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} \quad \text{(normalize)} \\
y_i &= \gamma \hat{x}_i + \beta \quad \text{(scale and shift)}
\end{align}
$$

Where:
- $\mu, \sigma^2$: computed **per sample** across its features
- $\epsilon$: small constant for numerical stability (e.g., 1e-5)
- $\gamma, \beta$: learnable parameters (same shape as features)

### Comparison with Batch Normalization

| Aspect | Batch Normalization | Layer Normalization |
|--------|---------------------|---------------------|
| **Normalization axis** | Across batch (samples) | Across features |
| **Statistics** | $\mu, \sigma$ per feature | $\mu, \sigma$ per sample |
| **Batch dependency** | ✅ Yes (couples samples) | ❌ No (independent) |
| **Learnable params** | $\gamma, \beta$ per feature | $\gamma, \beta$ per feature |
| **Train vs Test** | Different (running stats) | Same (no running stats) |

Let's implement this step by step!

---
## Part 4: Manual Implementation - From Scratch

The best way to understand LayerNorm is to implement it ourselves. We'll break it down into clear steps.

In [ ]:
def layer_norm_manual(x, gamma, beta, eps=1e-5):
    """
    Manual Layer Normalization implementation.
    
    Args:
        x: Input tensor of shape (batch_size, features)
        gamma: Scale parameter of shape (features,)
        beta: Shift parameter of shape (features,)
        eps: Small constant for numerical stability
    
    Returns:
        Normalized tensor of same shape as x
    """
    # Step 1: Compute mean across features (dim=1)
    # Result shape: (batch_size, 1)
    mean = x.mean(dim=1, keepdim=True)
    
    # Step 2: Compute variance across features (dim=1)
    # Result shape: (batch_size, 1)
    var = x.var(dim=1, keepdim=True, unbiased=False)
    
    # Step 3: Normalize
    # Broadcasting: (batch_size, features) - (batch_size, 1) = (batch_size, features)
    x_norm = (x - mean) / torch.sqrt(var + eps)
    
    # Step 4: Scale and shift
    # gamma and beta broadcast across batch dimension
    out = gamma * x_norm + beta
    
    return out, x_norm, mean, var

# Test on our example
batch_size, features = 4, 3
test_input = torch.randn(batch_size, features) * 10  # Random values with large scale

# Initialize learnable parameters
gamma = torch.ones(features)   # Scale starts at 1
beta = torch.zeros(features)   # Shift starts at 0

output, normalized, means, variances = layer_norm_manual(test_input, gamma, beta)

print("Input:")
print(test_input)
print(f"\nInput statistics per sample:")
for i in range(batch_size):
    print(f"  Sample {i}: mean={test_input[i].mean():.3f}, std={test_input[i].std():.3f}")

print("\nNormalized (before scale/shift):")
print(normalized)
print(f"\nNormalized statistics per sample:")
for i in range(batch_size):
    print(f"  Sample {i}: mean={normalized[i].mean():.3f}, std={normalized[i].std():.3f}")

print("\n✨ Notice: Each sample now has mean≈0, std≈1 across its features!")

### Visualizing the Transformation

In [ ]:
# Create visualization of normalization process
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i in range(min(3, batch_size)):  # Visualize first 3 samples
    sample_before = test_input[i].numpy()
    sample_after = normalized[i].numpy()
    
    # Before normalization
    axes[0, i].bar(range(features), sample_before, color='coral', alpha=0.7)
    axes[0, i].axhline(y=0, color='k', linestyle='--', alpha=0.3)
    axes[0, i].set_title(f'Sample {i}: Before\nmean={sample_before.mean():.2f}, std={sample_before.std():.2f}',
                        fontweight='bold')
    axes[0, i].set_ylabel('Value')
    axes[0, i].set_xlabel('Feature')
    axes[0, i].grid(True, alpha=0.3)
    
    # After normalization
    axes[1, i].bar(range(features), sample_after, color='lightblue', alpha=0.7)
    axes[1, i].axhline(y=0, color='k', linestyle='--', alpha=0.3)
    axes[1, i].set_title(f'Sample {i}: After\nmean={sample_after.mean():.2f}, std={sample_after.std():.2f}',
                        fontweight='bold')
    axes[1, i].set_ylabel('Value')
    axes[1, i].set_xlabel('Feature')
    axes[1, i].set_ylim(-3, 3)  # Typical normalized range
    axes[1, i].grid(True, alpha=0.3)

plt.suptitle('Layer Normalization: Per-Sample Feature Normalization', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("🔑 Key Observation:")
print("   Each sample is transformed independently!")
print("   All features within a sample are brought to similar scale.")

**🤔 Reflection Question:**

Why do we need learnable parameters ($\gamma$, $\beta$) if we just normalized everything?

<details>
<summary>Click to reveal answer</summary>

Great question! Normalization forces mean=0, std=1, but this might be too restrictive. The network might need features with different scales/offsets for optimal learning. $\gamma$ and $\beta$ let the network learn the best scale and shift for each feature. In the extreme case, the network can even learn to "undo" the normalization if that's optimal (by setting $\gamma = \sqrt{\sigma^2}$ and $\beta = \mu$).

</details>

---
## Part 5: Distribution Visualization - Before and After

Let's create a larger dataset and visualize how LayerNorm affects distributions.

In [ ]:
# Create a larger batch with features at different scales
n_samples = 1000
n_features = 5

# Each feature has different mean and scale
large_batch = torch.randn(n_samples, n_features)
large_batch[:, 0] = large_batch[:, 0] * 0.1 + 0.5    # Feature 0: small scale, offset
large_batch[:, 1] = large_batch[:, 1] * 5.0 - 2.0    # Feature 1: large scale, negative offset
large_batch[:, 2] = large_batch[:, 2] * 1.0          # Feature 2: standard normal
large_batch[:, 3] = large_batch[:, 3] * 10.0 + 10.0  # Feature 3: very large scale and offset
large_batch[:, 4] = large_batch[:, 4] * 0.5 - 5.0    # Feature 4: small scale, large negative offset

# Apply LayerNorm
gamma_large = torch.ones(n_features)
beta_large = torch.zeros(n_features)
normalized_large, _, _, _ = layer_norm_manual(large_batch, gamma_large, beta_large)

# Visualize distributions
fig, axes = plt.subplots(2, n_features, figsize=(18, 8))

for feat_idx in range(n_features):
    # Before normalization
    axes[0, feat_idx].hist(large_batch[:, feat_idx].numpy(), bins=50, 
                           color='coral', alpha=0.7, edgecolor='black')
    axes[0, feat_idx].set_title(f'Feature {feat_idx}\nBefore LayerNorm', fontweight='bold')
    axes[0, feat_idx].set_ylabel('Frequency')
    mean_val = large_batch[:, feat_idx].mean()
    std_val = large_batch[:, feat_idx].std()
    axes[0, feat_idx].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'μ={mean_val:.2f}')
    axes[0, feat_idx].legend()
    
    # After normalization  
    axes[1, feat_idx].hist(normalized_large[:, feat_idx].numpy(), bins=50,
                           color='lightblue', alpha=0.7, edgecolor='black')
    axes[1, feat_idx].set_title(f'Feature {feat_idx}\nAfter LayerNorm', fontweight='bold')
    axes[1, feat_idx].set_ylabel('Frequency')
    axes[1, feat_idx].set_xlabel('Value')
    axes[1, feat_idx].set_xlim(-4, 4)
    mean_val = normalized_large[:, feat_idx].mean()
    std_val = normalized_large[:, feat_idx].std()
    axes[1, feat_idx].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'μ={mean_val:.2f}')
    axes[1, feat_idx].legend()

plt.suptitle('Layer Normalization Effect on Feature Distributions', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("\n📊 Distribution Analysis:")
print("\nBefore LayerNorm (per feature stats):")
for i in range(n_features):
    print(f"  Feature {i}: mean={large_batch[:, i].mean():.3f}, std={large_batch[:, i].std():.3f}")

print("\nAfter LayerNorm (per feature stats):")
for i in range(n_features):
    print(f"  Feature {i}: mean={normalized_large[:, i].mean():.3f}, std={normalized_large[:, i].std():.3f}")

print("\n🔑 Insight: Features are NOT normalized to mean=0, std=1 globally!")
print("   LayerNorm normalizes WITHIN each sample, not across the batch.")
print("   The distributions above aggregate statistics across samples.")

### Per-Sample Distribution Check

Let's verify that each individual sample has mean≈0, std≈1:

In [ ]:
# Check statistics for individual samples
sample_means = normalized_large.mean(dim=1)  # Mean across features for each sample
sample_stds = normalized_large.std(dim=1)    # Std across features for each sample

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of per-sample means
mean_vals = sample_means.numpy()
axes[0].hist(mean_vals, bins=max(10, len(np.unique(mean_vals))//10), color='green', alpha=0.7, edgecolor='black')
axes[0].axvline(0, color='red', linestyle='--', linewidth=2, label='Target (0)')
axes[0].set_title('Distribution of Per-Sample Means', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Mean Value')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribution of per-sample stds
std_vals = sample_stds.numpy()
axes[1].hist(std_vals, bins=max(10, len(np.unique(std_vals))//10), color='purple', alpha=0.7, edgecolor='black')
axes[1].axvline(1, color='red', linestyle='--', linewidth=2, label='Target (1)')
axes[1].set_title('Distribution of Per-Sample Standard Deviations', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Std Value')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Per-sample statistics:")
print(f"  Mean: {sample_means.mean():.6f} ± {sample_means.std():.6f}")
print(f"  Std:  {sample_stds.mean():.6f} ± {sample_stds.std():.6f}")
print("\n✅ Each sample has mean≈0 and std≈1 across its features!")

---
## Part 6: PyTorch Implementation - Using nn.LayerNorm

Now let's use PyTorch's built-in LayerNorm and verify it matches our implementation.

In [ ]:
# Create LayerNorm layer
layer_norm = nn.LayerNorm(n_features)

# Important: Set parameters to match our manual implementation
with torch.no_grad():
    layer_norm.weight.fill_(1.0)  # gamma = 1
    layer_norm.bias.fill_(0.0)     # beta = 0

# Apply PyTorch LayerNorm
pytorch_output = layer_norm(large_batch)

# Compare with our manual implementation
diff = (pytorch_output - normalized_large).abs()

print("Comparison: Manual vs PyTorch LayerNorm")
print(f"  Maximum difference: {diff.max():.10f}")
print(f"  Mean difference: {diff.mean():.10f}")
print("\n✅ Our implementation matches PyTorch! (differences due to numerical precision)")

# Show first few samples
print("\nFirst 3 samples comparison:")
for i in range(3):
    print(f"\nSample {i}:")
    print(f"  Manual:  {normalized_large[i].numpy()}")
    print(f"  PyTorch: {pytorch_output[i].detach().numpy()}")
    print(f"  Diff:    {diff[i].detach().numpy()}")

### Exploring LayerNorm Parameters

In [ ]:
# Create LayerNorm with learnable parameters
ln = nn.LayerNorm(n_features)

print("LayerNorm Parameters:")
print(f"  Weight (gamma): shape={ln.weight.shape}, values={ln.weight.data}")
print(f"  Bias (beta):    shape={ln.bias.shape}, values={ln.bias.data}")

print("\nKey attributes:")
print(f"  Normalized shape: {ln.normalized_shape}")
print(f"  Epsilon: {ln.eps}")
print(f"  Elementwise affine: {ln.elementwise_affine}")

# Test with different gamma and beta
with torch.no_grad():
    ln.weight[:] = torch.tensor([0.5, 1.0, 1.5, 2.0, 2.5])  # Different scales per feature
    ln.bias[:] = torch.tensor([-1.0, -0.5, 0.0, 0.5, 1.0])  # Different shifts per feature

test_sample = torch.randn(1, n_features) * 10
output = ln(test_sample)

print("\nEffect of learnable parameters:")
print(f"  Input:  {test_sample[0].numpy()}")
print(f"  Output: {output[0].detach().numpy()}")
print(f"  Gamma:  {ln.weight.data.numpy()}")
print(f"  Beta:   {ln.bias.data.numpy()}")
print("\n🔑 Gamma scales and Beta shifts each feature differently!")

---
## Part 7: Training Dynamics - Does LayerNorm Help?

Let's train a simple MLP with and without LayerNorm to see the effect on training.

In [ ]:
# Create a simple classification dataset
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Generate synthetic dataset
X, y = make_classification(n_samples=2000, n_features=20, n_informative=15,
                          n_redundant=5, n_classes=2, random_state=42)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.LongTensor(y_test)

# Create dataloaders
train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Dataset created:")
print(f"  Train: {len(X_train)} samples")
print(f"  Test:  {len(X_test)} samples")
print(f"  Features: {X_train.shape[1]}")
print(f"  Classes: {len(np.unique(y))}")

Train the model and monitor progress.

In [ ]:
# Define two MLPs: one with LayerNorm, one without
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, use_layer_norm=False):
        super().__init__()
        self.use_layer_norm = use_layer_norm
        
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        
        if use_layer_norm:
            self.ln1 = nn.LayerNorm(hidden_dim)
            self.ln2 = nn.LayerNorm(hidden_dim)
    
    def forward(self, x):
        x = self.fc1(x)
        if self.use_layer_norm:
            x = self.ln1(x)
        x = F.relu(x)
        
        x = self.fc2(x)
        if self.use_layer_norm:
            x = self.ln2(x)
        x = F.relu(x)
        
        x = self.fc3(x)
        return x

# Training function
def train_model(model, train_loader, test_loader, epochs=30, lr=0.01):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    train_losses = []
    test_losses = []
    train_accs = []
    test_accs = []
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += y_batch.size(0)
            train_correct += predicted.eq(y_batch).sum().item()
        
        # Testing
        model.eval()
        test_loss = 0
        test_correct = 0
        test_total = 0
        
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                
                test_loss += loss.item()
                _, predicted = outputs.max(1)
                test_total += y_batch.size(0)
                test_correct += predicted.eq(y_batch).sum().item()
        
        train_losses.append(train_loss / len(train_loader))
        test_losses.append(test_loss / len(test_loader))
        train_accs.append(100. * train_correct / train_total)
        test_accs.append(100. * test_correct / test_total)
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} - "
                  f"Train Loss: {train_losses[-1]:.4f}, Test Loss: {test_losses[-1]:.4f}, "
                  f"Train Acc: {train_accs[-1]:.2f}%, Test Acc: {test_accs[-1]:.2f}%")
    
    return train_losses, test_losses, train_accs, test_accs

print("Models defined. Ready to train!")

Train the model and monitor progress.

In [ ]:
# Train both models
set_seed(42)
print("Training WITHOUT LayerNorm...")
model_no_ln = MLP(20, 64, 2, use_layer_norm=False)
train_loss_no_ln, test_loss_no_ln, train_acc_no_ln, test_acc_no_ln = train_model(
    model_no_ln, train_loader, test_loader, epochs=30, lr=0.01
)

print("\nTraining WITH LayerNorm...")
set_seed(42)
model_with_ln = MLP(20, 64, 2, use_layer_norm=True)
train_loss_with_ln, test_loss_with_ln, train_acc_with_ln, test_acc_with_ln = train_model(
    model_with_ln, train_loader, test_loader, epochs=30, lr=0.01
)

print("\n✅ Training complete!")

Visualize the results with multiple plots for comparison.

In [ ]:
# Visualize training dynamics
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

epochs_range = range(1, len(train_loss_no_ln) + 1)

# Loss curves
axes[0].plot(epochs_range, train_loss_no_ln, 'o-', label='No LN (Train)', color='coral', alpha=0.7)
axes[0].plot(epochs_range, test_loss_no_ln, 's-', label='No LN (Test)', color='coral')
axes[0].plot(epochs_range, train_loss_with_ln, 'o-', label='With LN (Train)', color='lightblue', alpha=0.7)
axes[0].plot(epochs_range, test_loss_with_ln, 's-', label='With LN (Test)', color='lightblue')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Test Loss', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(epochs_range, train_acc_no_ln, 'o-', label='No LN (Train)', color='coral', alpha=0.7)
axes[1].plot(epochs_range, test_acc_no_ln, 's-', label='No LN (Test)', color='coral')
axes[1].plot(epochs_range, train_acc_with_ln, 'o-', label='With LN (Train)', color='lightblue', alpha=0.7)
axes[1].plot(epochs_range, test_acc_with_ln, 's-', label='With LN (Test)', color='lightblue')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Training and Test Accuracy', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Final Results:")
print(f"\nWithout LayerNorm:")
print(f"  Train Loss: {train_loss_no_ln[-1]:.4f}, Test Loss: {test_loss_no_ln[-1]:.4f}")
print(f"  Train Acc:  {train_acc_no_ln[-1]:.2f}%, Test Acc:  {test_acc_no_ln[-1]:.2f}%")

print(f"\nWith LayerNorm:")
print(f"  Train Loss: {train_loss_with_ln[-1]:.4f}, Test Loss: {test_loss_with_ln[-1]:.4f}")
print(f"  Train Acc:  {train_acc_with_ln[-1]:.2f}%, Test Acc:  {test_acc_with_ln[-1]:.2f}%")

print("\n🔑 Observation: LayerNorm often leads to smoother training and better convergence!")

---
## Part 8: The Critical Experiment - Batch Size Independence

This is the **most important difference** between BatchNorm and LayerNorm!

### Hypothesis:
- **BatchNorm**: Performance degrades with very small batches (unstable statistics)
- **LayerNorm**: Performance should be **independent** of batch size

Let's test this!

In [ ]:
# Define models with BatchNorm for comparison
class MLPWithBatchNorm(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.fc3(x)
        return x

# Test different batch sizes
batch_sizes = [4, 8, 32, 128]  # Skip batch_size=1 for BatchNorm (will fail)
results_bn = {}
results_ln = {}

# First test LayerNorm with batch_size=1 separately to demonstrate independence
print(f"\n{'='*60}")
print(f"Testing LayerNorm with batch_size=1 (BatchNorm can't do this!)")
print(f"{'='*60}")
train_loader_1 = DataLoader(train_dataset, batch_size=1, shuffle=True)
test_loader_1 = DataLoader(test_dataset, batch_size=1, shuffle=False)
set_seed(42)
model_ln_1 = MLP(20, 64, 2, use_layer_norm=True)
_, _, _, test_acc_ln_1 = train_model(
    model_ln_1, train_loader_1, test_loader_1, epochs=20, lr=0.01
)
results_ln[1] = test_acc_ln_1[-1]
print(f"\nLayerNorm with batch_size=1: {results_ln[1]:.2f}%")
print("✅ LayerNorm works perfectly with batch_size=1!")

# Now test both with larger batch sizes
for bs in batch_sizes:
    print(f"\n{'='*60}")
    print(f"Testing with batch size: {bs}")
    print(f"{'='*60}")
    
    # Create dataloaders with specific batch size
    train_loader_bs = DataLoader(train_dataset, batch_size=bs, shuffle=True)
    test_loader_bs = DataLoader(test_dataset, batch_size=bs, shuffle=False)
    
    # Train with BatchNorm
    print(f"\nTraining with BatchNorm (batch_size={bs})...")
    set_seed(42)
    model_bn = MLPWithBatchNorm(20, 64, 2)
    _, _, _, test_acc_bn = train_model(
        model_bn, train_loader_bs, test_loader_bs, epochs=20, lr=0.01
    )
    results_bn[bs] = test_acc_bn[-1]
    
    # Train with LayerNorm
    print(f"\nTraining with LayerNorm (batch_size={bs})...")
    set_seed(42)
    model_ln = MLP(20, 64, 2, use_layer_norm=True)
    _, _, _, test_acc_ln = train_model(
        model_ln, train_loader_bs, test_loader_bs, epochs=20, lr=0.01
    )
    results_ln[bs] = test_acc_ln[-1]
    
    print(f"\nResults for batch_size={bs}:")
    print(f"  BatchNorm:  {results_bn[bs]:.2f}%")
    print(f"  LayerNorm:  {results_ln[bs]:.2f}%")

print("\n✅ Batch size experiments complete!")

Visualize the results with multiple plots for comparison.

In [ ]:
# Visualize batch size effect
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

# Prepare data for visualization
all_batch_sizes = [1] + batch_sizes  # Include batch_size=1
bn_accs = [None] + [results_bn[bs] for bs in batch_sizes]  # BatchNorm can't do bs=1
ln_accs = [results_ln[bs] for bs in all_batch_sizes]

x_pos = np.arange(len(all_batch_sizes))
width = 0.35

# Plot BatchNorm bars (skip first one for bs=1)
bn_x = x_pos[1:]  # Skip first position
bn_vals = [results_bn[bs] for bs in batch_sizes]
ax.bar(bn_x - width/2, bn_vals, width, label='BatchNorm', color='coral', alpha=0.8)

# Plot LayerNorm bars (all batch sizes)
ax.bar(x_pos + width/2, ln_accs, width, label='LayerNorm', color='lightblue', alpha=0.8)

# Add special marker for batch_size=1 (LayerNorm only)
ax.text(0 - width/2, 50, '✗\nCan\'t train', ha='center', va='center', 
        fontsize=10, color='red', fontweight='bold')

ax.set_xlabel('Batch Size', fontsize=14, fontweight='bold')
ax.set_ylabel('Test Accuracy (%)', fontsize=14, fontweight='bold')
ax.set_title('Batch Size Independence: LayerNorm vs BatchNorm', fontsize=16, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(all_batch_sizes)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

# Calculate y-axis range
all_accs = [acc for acc in ln_accs if acc is not None] + bn_vals
ax.set_ylim(min(all_accs) - 5, 100)

# Add value labels on bars
for i, ln_acc in enumerate(ln_accs):
    ax.text(i + width/2, ln_acc + 1, f'{ln_acc:.1f}%', ha='center', fontsize=10)

for i, bn_acc in enumerate(bn_vals):
    ax.text(i + 1 - width/2, bn_acc + 1, f'{bn_acc:.1f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print("\n🔑 KEY INSIGHT:")
print("   LayerNorm performance is stable across ALL batch sizes (including 1)!")
print("   BatchNorm can't train with batch_size=1 (fails during training).")
print("   Even with larger batches, LayerNorm shows more consistent performance.")
print("\n   This is WHY LayerNorm is preferred for:")
print("     - Online learning (batch_size=1)")
print("     - Reinforcement learning (small batches)")
print("     - Sequence models (variable batch sizes)")
print("     - Generation tasks (often batch_size=1)")

**🤔 Reflection Question:**

Why does BatchNorm struggle with batch_size=1, but LayerNorm works fine?

<details>
<summary>Click to reveal answer</summary>

With batch_size=1, BatchNorm has only ONE sample to compute mean and variance from, making the statistics meaningless (variance would be 0!). LayerNorm, however, computes statistics across features within that single sample, which can be many (e.g., 64 hidden units), providing stable statistics.

</details>

---
## Part 9: Sequential Data and Transformers

LayerNorm is the **normalization of choice** for Transformers and sequence models. Let's understand why.

### Why LayerNorm for Sequences?

1. **Variable sequence lengths**: Different samples have different lengths
2. **Position independence**: Each position is somewhat independent
3. **Per-token processing**: We want to normalize each token's features
4. **Batch size independence**: Critical for generation (often batch_size=1)

Let's demonstrate with a simple sequence model:

In [ ]:
# Create a simple sequence classification task
# Task: Classify sequences based on their mean value

def create_sequence_data(n_samples=1000, seq_len_range=(10, 50), n_features=16):
    """Create synthetic sequence classification data."""
    sequences = []
    labels = []
    
    for _ in range(n_samples):
        # Random sequence length
        seq_len = np.random.randint(seq_len_range[0], seq_len_range[1])
        
        # Generate sequence
        if np.random.rand() > 0.5:
            # Class 0: low mean
            seq = torch.randn(seq_len, n_features) - 1.0
            label = 0
        else:
            # Class 1: high mean
            seq = torch.randn(seq_len, n_features) + 1.0
            label = 1
        
        sequences.append(seq)
        labels.append(label)
    
    return sequences, labels

# Create data
train_seqs, train_labels = create_sequence_data(n_samples=800)
test_seqs, test_labels = create_sequence_data(n_samples=200)

print(f"Created sequence dataset:")
print(f"  Train sequences: {len(train_seqs)}")
print(f"  Test sequences: {len(test_seqs)}")
print(f"  Sequence length range: 10-50")
print(f"  Features per timestep: 16")
print(f"\nExample sequence shapes:")
for i in range(3):
    print(f"  Sequence {i}: {train_seqs[i].shape}, label: {train_labels[i]}")

Prepare the data loaders for training and validation.

In [ ]:
# Simple sequence model with LayerNorm
class SequenceClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, use_layer_norm=True):
        super().__init__()
        self.use_layer_norm = use_layer_norm
        
        self.rnn = nn.GRU(input_dim, hidden_dim, batch_first=True)
        
        if use_layer_norm:
            self.ln = nn.LayerNorm(hidden_dim)
        
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        # x shape: (batch, seq_len, features)
        output, hidden = self.rnn(x)  # output: (batch, seq_len, hidden_dim)
        
        # Use last timestep
        last_output = output[:, -1, :]  # (batch, hidden_dim)
        
        if self.use_layer_norm:
            last_output = self.ln(last_output)
        
        logits = self.fc(last_output)
        return logits

# Collate function for variable-length sequences
def collate_sequences(batch):
    sequences, labels = zip(*batch)
    # Pad sequences to same length
    padded = nn.utils.rnn.pad_sequence(sequences, batch_first=True)
    labels = torch.tensor(labels)
    return padded, labels

# Create datasets and loaders
from torch.utils.data import Dataset

class SequenceDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

train_seq_dataset = SequenceDataset(train_seqs, train_labels)
test_seq_dataset = SequenceDataset(test_seqs, test_labels)

train_seq_loader = DataLoader(train_seq_dataset, batch_size=32, shuffle=True, collate_fn=collate_sequences)
test_seq_loader = DataLoader(test_seq_dataset, batch_size=32, shuffle=False, collate_fn=collate_sequences)

print("Sequence datasets ready!")

Visualize the results.

In [ ]:
# Train sequence models
print("Training sequence model WITHOUT LayerNorm...")
set_seed(42)
seq_model_no_ln = SequenceClassifier(16, 32, 2, use_layer_norm=False)
_, _, _, seq_acc_no_ln = train_model(
    seq_model_no_ln, train_seq_loader, test_seq_loader, epochs=20, lr=0.001
)

print("\nTraining sequence model WITH LayerNorm...")
set_seed(42)
seq_model_with_ln = SequenceClassifier(16, 32, 2, use_layer_norm=True)
_, _, _, seq_acc_with_ln = train_model(
    seq_model_with_ln, train_seq_loader, test_seq_loader, epochs=20, lr=0.001
)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(range(1, 21), seq_acc_no_ln, 'o-', label='Without LayerNorm', color='coral', linewidth=2)
plt.plot(range(1, 21), seq_acc_with_ln, 's-', label='With LayerNorm', color='lightblue', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Test Accuracy (%)', fontsize=12)
plt.title('Sequence Classification: Effect of LayerNorm', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal Test Accuracy:")
print(f"  Without LayerNorm: {seq_acc_no_ln[-1]:.2f}%")
print(f"  With LayerNorm:    {seq_acc_with_ln[-1]:.2f}%")
print("\n🔑 LayerNorm helps stabilize training for sequence models!")

### Why Transformers Use LayerNorm

In Transformers, LayerNorm is applied:
1. **After self-attention** (or before, in Pre-LN)
2. **After feed-forward layers** (or before, in Pre-LN)

Shape conventions:
- Input: `(batch_size, seq_len, d_model)`
- LayerNorm operates on `d_model` dimension (features)
- Each token is normalized independently

This is perfect because:
- Handles variable sequence lengths naturally
- Works with batch_size=1 during generation
- Stabilizes deep networks (Transformers can be 100+ layers!)

---
## Part 10: Layer Position - Pre-LN vs Post-LN

In Transformer architectures, **where** you place LayerNorm matters a lot!

### Post-LN (Original Transformer)
```
x = x + SelfAttention(x)
x = LayerNorm(x)
x = x + FeedForward(x)
x = LayerNorm(x)
```

### Pre-LN (Modern Preference)
```
x = x + SelfAttention(LayerNorm(x))
x = x + FeedForward(LayerNorm(x))
```

**Key difference**: Pre-LN normalizes **before** the layer, Post-LN normalizes **after**.

### Why does this matter?

Pre-LN provides:
1. **Better gradient flow** (gradients don't explode/vanish as easily)
2. **Easier training** of very deep networks
3. **Less need for learning rate warmup**

Let's visualize this with a simple experiment:

In [ ]:
# Simple demonstration of Pre-LN vs Post-LN
class TransformerBlock(nn.Module):
    """Simplified transformer block for demonstration."""
    def __init__(self, d_model, n_heads, use_pre_ln=True):
        super().__init__()
        self.use_pre_ln = use_pre_ln
        
        self.attention = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model)
        )
        
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
    
    def forward(self, x):
        if self.use_pre_ln:
            # Pre-LN: normalize before sub-layer
            attn_out, _ = self.attention(self.ln1(x), self.ln1(x), self.ln1(x))
            x = x + attn_out
            x = x + self.ffn(self.ln2(x))
        else:
            # Post-LN: normalize after sub-layer
            attn_out, _ = self.attention(x, x, x)
            x = self.ln1(x + attn_out)
            x = self.ln2(x + self.ffn(x))
        
        return x

class SimpleTransformer(nn.Module):
    def __init__(self, d_model, n_heads, n_layers, output_dim, use_pre_ln=True):
        super().__init__()
        self.embedding = nn.Linear(16, d_model)  # Simple embedding
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, use_pre_ln) for _ in range(n_layers)
        ])
        self.fc_out = nn.Linear(d_model, output_dim)
    
    def forward(self, x):
        # x: (batch, seq_len, features)
        x = self.embedding(x)
        
        for block in self.blocks:
            x = block(x)
        
        # Global average pooling
        x = x.mean(dim=1)
        return self.fc_out(x)

print("Transformer models defined!")

Visualize the results.

In [ ]:
# Train both variants
print("Training with Post-LN (original)...")
set_seed(42)
model_post_ln = SimpleTransformer(d_model=32, n_heads=4, n_layers=3, output_dim=2, use_pre_ln=False)
_, _, _, acc_post_ln = train_model(
    model_post_ln, train_seq_loader, test_seq_loader, epochs=20, lr=0.0001
)

print("\nTraining with Pre-LN (modern)...")
set_seed(42)
model_pre_ln = SimpleTransformer(d_model=32, n_heads=4, n_layers=3, output_dim=2, use_pre_ln=True)
_, _, _, acc_pre_ln = train_model(
    model_pre_ln, train_seq_loader, test_seq_loader, epochs=20, lr=0.0001
)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(range(1, 21), acc_post_ln, 'o-', label='Post-LN (original)', color='coral', linewidth=2)
plt.plot(range(1, 21), acc_pre_ln, 's-', label='Pre-LN (modern)', color='lightblue', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Test Accuracy (%)', fontsize=12)
plt.title('Pre-LN vs Post-LN in Transformers', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal Test Accuracy:")
print(f"  Post-LN: {acc_post_ln[-1]:.2f}%")
print(f"  Pre-LN:  {acc_pre_ln[-1]:.2f}%")
print("\n🔑 Pre-LN often provides more stable training!")
print("   This is why modern Transformers (GPT-3, etc.) use Pre-LN.")

---
## Part 11: Pros and Cons - When to Use What?

Now that we understand LayerNorm deeply, let's create a clear decision framework.

### Layer Normalization

**✅ Pros:**
1. **Batch size independent** - Works great with any batch size, including 1
2. **No train/test discrepancy** - Same computation during training and inference
3. **Perfect for sequences** - Handles variable-length sequences naturally
4. **RNN/Transformer friendly** - Normalizes per timestep/token
5. **Online learning** - Can process samples one at a time
6. **Simpler inference** - No need to track running statistics

**❌ Cons:**
1. **Requires enough features** - Needs multiple features per sample for stable statistics
2. **Different semantics** - Normalizes across features, not across batch
3. **Less effective for CNNs** - Spatial structure not well-exploited (use GroupNorm instead)
4. **Computational cost** - Per-sample computation (though usually not a bottleneck)

### Batch Normalization

**✅ Pros:**
1. **Great for CNNs** - Exploits spatial structure well
2. **Regularization effect** - Noise from batch statistics acts as regularizer
3. **Feature independence** - Normalizes each feature channel independently
4. **Empirically strong** - Very effective for computer vision

**❌ Cons:**
1. **Batch size dependent** - Poor with small batches
2. **Train/test discrepancy** - Different behavior during training vs inference
3. **Awkward for sequences** - Doesn't handle variable lengths well
4. **Not for online learning** - Can't process one sample at a time effectively

### Decision Framework

In [ ]:
# Create decision tree visualization
from IPython.display import Markdown

decision_tree = """
## 🎯 Which Normalization Should I Use?

### Start Here:

**1. What type of data?**
   - **Sequences (text, time-series)** → LayerNorm ✓
   - **Images (CNNs)** → Continue to Q2
   - **Tabular/MLP** → Continue to Q2

**2. What's your batch size?**
   - **Small (< 8) or variable** → LayerNorm ✓
   - **Large (≥ 32)** → Continue to Q3

**3. Training scenario?**
   - **Online learning / streaming** → LayerNorm ✓
   - **Standard mini-batch** → Continue to Q4

**4. Architecture type?**
   - **Transformer** → LayerNorm (Pre-LN) ✓
   - **RNN/LSTM/GRU** → LayerNorm ✓
   - **CNN** → BatchNorm or GroupNorm ✓
   - **MLP** → Either works, try both!

**5. Do you need identical train/test behavior?**
   - **Yes (important for debugging/reproducibility)** → LayerNorm ✓
   - **No** → Either works

### Quick Reference Table:

| Use Case | Recommended | Why |
|----------|-------------|-----|
| **Transformers** | LayerNorm (Pre-LN) | Batch independence, deep networks |
| **RNNs** | LayerNorm | Handles sequences, variable length |
| **CNNs** | BatchNorm or GroupNorm | Spatial structure, large batches |
| **MLPs** | Either | Experiment with both |
| **Reinforcement Learning** | LayerNorm | Small/variable batches |
| **Online Learning** | LayerNorm | Batch size = 1 |
| **Large-batch training** | BatchNorm | Strong regularization effect |
| **Small-batch training** | LayerNorm | Stable statistics |

### Special Cases:

- **GroupNorm**: Middle ground for CNNs when batch size is small
- **RMSNorm**: Simplified LayerNorm (no mean centering) - faster, often just as good
- **InstanceNorm**: For style transfer (normalizes each instance independently)
"""

display(Markdown(decision_tree))

---
## Part 12: Common Pitfalls and Debugging

Let's explore common mistakes and how to debug them.

### Pitfall 1: Wrong Normalization Dimensions

In [ ]:
# Common mistake: normalizing wrong dimension
batch = torch.randn(4, 8, 16)  # (batch_size, seq_len, features)

print("Input shape: (batch_size=4, seq_len=8, features=16)")
print("\n" + "="*60)

# CORRECT: Normalize across features (last dimension)
ln_correct = nn.LayerNorm(16)  # normalized_shape = features
output_correct = ln_correct(batch)
print("✅ CORRECT: LayerNorm(16) - normalizes across features")
print(f"   Output shape: {output_correct.shape}")
print(f"   Stats for sample 0, timestep 0: mean={output_correct[0, 0].mean():.4f}, std={output_correct[0, 0].std():.4f}")

print("\n" + "="*60)

# WRONG: Normalizing wrong dimension
print("❌ WRONG: LayerNorm(8) - normalizes across timesteps (usually not what you want!)")
ln_wrong = nn.LayerNorm(8)  # This would normalize across seq_len!
try:
    # This will fail because last dimension is 16, not 8
    output_wrong = ln_wrong(batch)
except RuntimeError as e:
    print(f"   Error: {e}")

print("\n🔑 Rule: normalized_shape should match the dimensions you want to normalize across.")
print("   For shape (batch, seq, features), use LayerNorm(features).")

### Pitfall 2: Too Few Features

In [ ]:
# LayerNorm needs enough features for stable statistics
print("Effect of number of features on LayerNorm stability:\n")

for n_features in [2, 5, 10, 50, 100]:
    test_data = torch.randn(100, n_features) * 10  # Large scale
    ln = nn.LayerNorm(n_features)
    normalized = ln(test_data)
    
    # Check how well statistics match target
    sample_means = normalized.mean(dim=1)
    sample_stds = normalized.std(dim=1)
    
    mean_of_means = sample_means.mean()
    std_of_means = sample_means.std()
    mean_of_stds = sample_stds.mean()
    std_of_stds = sample_stds.std()
    
    print(f"n_features={n_features:3d}: mean={mean_of_means:+.4f}±{std_of_means:.4f}, std={mean_of_stds:.4f}±{std_of_stds:.4f}")

print("\n🔑 Insight: With very few features (e.g., 2), statistics are less stable.")
print("   LayerNorm works best with ≥10 features.")

### Pitfall 3: Forgetting Epsilon

In [ ]:
# What happens without epsilon?
test_input = torch.ones(4, 10) * 5.0  # All features have same value!

print("Input: All features in each sample have the same value (5.0)")
print(f"Variance: {test_input[0].var():.10f}")

# Without epsilon, we'd divide by zero!
mean = test_input.mean(dim=1, keepdim=True)
var = test_input.var(dim=1, keepdim=True, unbiased=False)

print(f"\nWithout epsilon:")
try:
    normalized_no_eps = (test_input - mean) / torch.sqrt(var)
    print(f"  Result: {normalized_no_eps[0]}")  # Will be nan!
except:
    print("  Error: Division by zero!")

# With epsilon
eps = 1e-5
normalized_with_eps = (test_input - mean) / torch.sqrt(var + eps)
print(f"\nWith epsilon (eps={eps}):")
print(f"  Result: {normalized_with_eps[0]}")

print("\n🔑 Epsilon prevents division by zero when variance is very small!")

### Debugging Checklist

When LayerNorm isn't working:

1. ✓ **Check shapes**: Does `normalized_shape` match your feature dimensions?
2. ✓ **Check statistics**: Are per-sample means ≈0 and stds ≈1?
3. ✓ **Check features**: Do you have enough features (≥10 recommended)?
4. ✓ **Check gradients**: Are gradients flowing properly? (use `.register_hook()`)
5. ✓ **Check placement**: Pre-LN or Post-LN? Try both!
6. ✓ **Check epsilon**: Default (1e-5) usually works, increase if instability
7. ✓ **Check initialization**: Are learnable params (γ, β) initialized reasonably?

---
## Part 13: Advanced Topics

### RMSNorm - Simpler is Better?

In [ ]:
# RMSNorm: Root Mean Square Normalization
# Removes mean centering step - just normalize by RMS

class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization.
    
    Simplified version of LayerNorm that:
    - Skips mean centering (no mean subtraction)
    - Only normalizes by RMS (root mean square)
    - Faster and often just as effective
    
    Used in: LLaMA, Gopher, and other modern LLMs
    """
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(normalized_shape))
    
    def forward(self, x):
        # Compute RMS (no mean subtraction!)
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        
        # Normalize and scale
        x_norm = x / rms
        return self.weight * x_norm

# Compare LayerNorm vs RMSNorm
test_data = torch.randn(4, 16) * 5.0

ln = nn.LayerNorm(16)
rms = RMSNorm(16)

# Set same weights for fair comparison
with torch.no_grad():
    rms.weight.copy_(ln.weight)
    ln.bias.zero_()

output_ln = ln(test_data)
output_rms = rms(test_data)

print("Comparison: LayerNorm vs RMSNorm\n")
print("Sample 0:")
print(f"  Input:      mean={test_data[0].mean():.3f}, std={test_data[0].std():.3f}")
print(f"  LayerNorm:  mean={output_ln[0].mean():.3f}, std={output_ln[0].std():.3f}")
print(f"  RMSNorm:    mean={output_rms[0].mean():.3f}, std={output_rms[0].std():.3f}")

print("\n🔑 Key Difference:")
print("   LayerNorm: Centers at 0 (mean ≈ 0)")
print("   RMSNorm:   Only normalizes scale (mean ≠ 0)")
print("\n   RMSNorm is simpler, faster, and often just as effective!")
print("   Used in many modern LLMs (LLaMA, etc.)")

### Group Normalization - Middle Ground for CNNs

In [ ]:
# Group Normalization: divides channels into groups
# Each group is normalized independently

# Example: CNN feature map
# Shape: (batch=2, channels=16, height=8, width=8)
feature_map = torch.randn(2, 16, 8, 8)

# Different normalization strategies
bn = nn.BatchNorm2d(16)          # Normalize across batch
ln = nn.LayerNorm([16, 8, 8])    # Normalize across channels, height, width
gn = nn.GroupNorm(4, 16)         # Normalize within groups (4 groups of 4 channels)
inorm = nn.InstanceNorm2d(16)    # Normalize each instance independently

output_bn = bn(feature_map)
output_ln = ln(feature_map)
output_gn = gn(feature_map)
output_inorm = inorm(feature_map)

print("CNN Feature Map Normalization Comparison:\n")
print(f"Input shape: {feature_map.shape}")
print(f"\nBatchNorm:     {output_bn.shape}")
print(f"LayerNorm:     {output_ln.shape}")
print(f"GroupNorm:     {output_gn.shape}")
print(f"InstanceNorm:  {output_inorm.shape}")

print("\n🔑 For CNNs:")
print("   BatchNorm:    Best for large batches")
print("   GroupNorm:    Best for small batches (middle ground)")
print("   LayerNorm:    Less common for CNNs (but used in Vision Transformers!)")
print("   InstanceNorm: Best for style transfer")

---
## Part 14: Hands-On Experiments

Now it's your turn! Try these experiments to cement your understanding.

### 🔬 Experiment 1: Implement RMSNorm and Compare

**Task**: Implement RMSNorm (provided above) and train a model comparing it to LayerNorm.

**Questions to explore**:
- Is RMSNorm faster?
- Does it achieve similar accuracy?
- How do the learned weights differ?

In [ ]:
# Your code here!
# Hint: Modify the MLP class to use RMSNorm instead of LayerNorm
# Train both and compare results



### 🔬 Experiment 2: LayerNorm with Different Numbers of Features

**Task**: Create synthetic data with varying numbers of features (2, 5, 10, 50, 100) and train models to see how LayerNorm performance changes.

**Questions to explore**:
- What's the minimum number of features for stable LayerNorm?
- How does training stability change with more features?
- Plot the relationship between n_features and final accuracy.

In [ ]:
# Your code here!



### 🔬 Experiment 3: Visualize Feature Correlations

**Task**: Visualize feature correlations before and after LayerNorm.

**Questions to explore**:
- Does LayerNorm affect feature correlations?
- How does this compare to BatchNorm?
- Create a correlation matrix heatmap for visualization.

In [ ]:
# Your code here!
# Hint: Use torch.corrcoef() to compute correlation matrix
# Use seaborn.heatmap() to visualize



### 🔬 Experiment 4: Gradient Flow Analysis

**Task**: Analyze gradient flow in deep networks with and without LayerNorm.

**Questions to explore**:
- Do gradients vanish/explode without LayerNorm?
- How does Pre-LN vs Post-LN affect gradient magnitudes?
- Plot gradient magnitudes across layers.

In [ ]:
# Your code here!
# Hint: Use .register_hook() to capture gradients



---
## Part 15: Summary and Key Takeaways

### 🎯 Core Concepts Mastered:

1. **LayerNorm normalizes across features** (not batch)
   - Each sample processed independently
   - Mean ≈ 0, std ≈ 1 **per sample**

2. **Batch size independence** is the killer feature
   - Works with batch_size=1 (BatchNorm doesn't)
   - Perfect for online learning, RL, generation

3. **Transformers use LayerNorm** for good reasons
   - Handles sequences naturally
   - Pre-LN is modern preference
   - Stabilizes very deep networks

4. **Decision framework**:
   - **Sequences → LayerNorm**
   - **Small batches → LayerNorm**
   - **Online learning → LayerNorm**
   - **CNNs + large batches → BatchNorm**
   - **CNNs + small batches → GroupNorm**

### 🔑 Most Important Insights:

1. LayerNorm's batch independence makes it ideal for:
   - Variable batch sizes
   - Inference with batch_size=1
   - Sequence models with variable lengths

2. The normalization axis matters:
   - BatchNorm: across samples (↓)
   - LayerNorm: across features (→)
   - This fundamental difference drives all other properties

3. Modern variants (RMSNorm) are simpler and often just as good

4. Placement matters: Pre-LN vs Post-LN affects training stability

### 📚 What's Next?

- Implement LayerNorm in your own projects
- Try RMSNorm for faster training
- Experiment with Pre-LN vs Post-LN in Transformers
- Compare normalization techniques on your specific tasks

### 🎓 You now have strong intuitions for:
- ✅ How LayerNorm works mathematically
- ✅ Why it's different from BatchNorm
- ✅ When to use each normalization technique
- ✅ How to implement and debug LayerNorm
- ✅ Modern variants and best practices

**Congratulations! 🎉**

You've built deep intuitions for Layer Normalization through theory, visualizations, and hands-on experiments!

---
## 📖 References and Further Reading

### Papers:
1. **Layer Normalization** (Ba et al., 2016): Original paper introducing LayerNorm
2. **Root Mean Square Layer Normalization** (Zhang & Sennrich, 2019): RMSNorm
3. **On Layer Normalization in the Transformer Architecture** (Xiong et al., 2020): Pre-LN vs Post-LN
4. **Group Normalization** (Wu & He, 2018): Alternative for CNNs

### Key Concepts:
- Batch Normalization (compare with this notebook!)
- Transformer architecture (where LayerNorm shines)
- Gradient flow in deep networks
- Numerical stability in deep learning

---

*This notebook was created to build strong intuitions through progressive learning, visualizations, and hands-on experimentation.*